# 06 — اختبار الـ Metrics (`prometheus_client`)

مش كفاية إن `/metrics` يرجّع 200. هنتأكد إن الأرقام **صح**:

| # | المحور | السؤال |
|---|---|---|
| 1 | Endpoint | الـ format صالح؟ HELP/TYPE؟ مفيش duplicates / NaN؟ |
| 2 | Inventory | إيه الـ metrics الموجودة؟ مين اتعرّف في الكود؟ |
| 3 | Naming lint | أسماء/وحدات/labels حسب الـ best practices؟ |
| 4 | HTTP counter | بيزيد **بالظبط** بعدد الـ requests؟ الـ status labels صح؟ |
| 5 | Histogram | buckets تراكمية؟ `+Inf == count`؟ الوحدة seconds فعلاً؟ |
| 6 | Cardinality | الـ path label بيستخدم template (`/files/{id}`) مش الـ UUID الخام؟ |
| 7 | Concurrency | الأرقام دقيقة تحت load؟ gauge الـ in-progress بيرجع لصفر؟ |
| 8 | Business metrics | ingestion / embedding / search / LLM بتتسجّل فعلاً بعد flow كامل؟ |
| 9 | Ops | scrape سريع؟ multi-process؟ الـ endpoint مكشوف؟ |

الـ detection تلقائي (بيدوّر على الـ counter/histogram بالـ labels `method` + `status`) فبيشتغل مع `prometheus-fastapi-instrumentator` أو middleware مكتوب بإيدك.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
sys.path.insert(0, str(pathlib.Path.cwd() / "notebooks"))
from nb_utils import *
env = setup()
print(env)
rec = Recorder("06_prometheus_metrics")

In [ ]:
import matplotlib.pyplot as plt
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
import uuid as _uuid

api = get_api(); app = api.app
r0 = scrape(api)
FAMS, FLAT = parse_metrics(r0.text) if r0.status_code == 200 else ([], {})
print("scrape status:", r0.status_code, "| families:", len(FAMS), "| series:", len(FLAT))

## 1) الـ Endpoint والـ Exposition format

In [ ]:
with rec.case("GET /metrics returns 200", "endpoint"):
    assert r0.status_code == 200, f"HTTP {r0.status_code} - is /metrics mounted? (make_asgi_app / Instrumentator().expose(app))"
with rec.case("content-type is the Prometheus text format", "endpoint") as c:
    ct = r0.headers.get("content-type", ""); c.note(ct); assert ct.startswith("text/plain") or "openmetrics" in ct, ct
with rec.case("payload parses with prometheus_client's own parser", "endpoint") as c:
    need(FAMS or None); c.note(f"{len(FAMS)} families, {len(FLAT)} series")
with rec.case("every family has # HELP and # TYPE", "endpoint") as c:
    need(FAMS or None)
    miss = [f.name for f in FAMS if f"# HELP {f.name}" not in r0.text or f"# TYPE {f.name}" not in r0.text]
    assert not miss, f"missing HELP/TYPE: {miss[:6]}"
with rec.case("payload ends with a newline", "endpoint"):
    need(FAMS or None); assert r0.text.endswith("\n")
with rec.case("no duplicated series in a single scrape", "endpoint") as c:
    need(FAMS or None)
    keys = [ln.rsplit(" ", 1)[0] for ln in r0.text.splitlines() if ln.strip() and not ln.startswith("#")]
    dup = {k for k in keys if keys.count(k) > 1}; assert not dup, f"duplicates: {list(dup)[:3]}"
with rec.case("no NaN, no negative counters", "endpoint") as c:
    need(FAMS or None)
    counters = {s.name for f in FAMS if f.type in ("counter", "histogram", "summary") for s in f.samples}
    bad = [(k[0], v) for k, v in FLAT.items() if math.isnan(v) or (k[0] in counters and v < 0 and not k[0].endswith("_sum"))]
    assert not bad, f"bad values: {bad[:4]}"

## 2) Inventory — إيه الموجود؟

In [ ]:
inv = []
for f in FAMS:
    labels = sorted({l for s in f.samples for l in s.labels if l != "le"})
    inv.append(dict(name=f.name, type=f.type, series=len(f.samples), labels=",".join(labels), kind="default" if f.name.startswith(DEFAULT_PREFIXES) else "app", help=f.documentation[:70]))
INV = pd.DataFrame(inv)
if len(INV):
    display(INV.sort_values(["kind", "name"]).reset_index(drop=True))
    print(INV.groupby(["kind", "type"]).size().unstack(fill_value=0))
rec.check("app-level (non-default) metrics exist", bool(len(INV)) and (INV.kind == "app").any(), f"{int((INV.kind == 'app').sum()) if len(INV) else 0} families", "inventory")
rec.check("default python_/process_ collectors enabled", bool(len(INV)) and (INV.kind == "default").any(), "", "inventory", level="warn")

# metric objects defined in YOUR code (found by scanning loaded modules)
from prometheus_client.metrics import MetricWrapperBase
found, seen = [], set()
for modname, mod in list(sys.modules.items()):
    if mod is None or modname.startswith(("prometheus_client", "IPython", "ipykernel", "_pytest")): continue
    try: items = list(vars(mod).items())
    except TypeError: continue
    for n, o in items:
        if isinstance(o, MetricWrapperBase) and id(o) not in seen:
            seen.add(id(o)); found.append(dict(module=modname, variable=n, metric=o._name, type=o._type, labels=",".join(o._labelnames), help=o._documentation[:50]))
print("\nMetric objects defined in code:"); display(pd.DataFrame(found))

## 3) Naming / label lint

In [ ]:
HIGH_CARD = {"user_id", "email", "file_id", "id", "uuid", "session_id", "request_id", "query", "filename", "token", "conversation_id", "project_id", "workspace_id", "chunk_id"}
for f in FAMS:
    if f.name.startswith(DEFAULT_PREFIXES): continue
    with rec.case(f"lint: {f.name}", "lint") as c:
        iss = []
        if not re.fullmatch(r"[a-z][a-z0-9_]*", f.name): iss.append("name is not snake_case")
        if f.type == "counter" and not any(s.name.endswith("_total") for s in f.samples if not s.name.endswith("_created")) and f.samples:
            iss.append("counter sample should end with _total")
        if f.type in ("histogram", "summary") and re.search(r"duration|latency|time", f.name) and "seconds" not in f.name:
            iss.append("time metric should be in base unit and end with _seconds")
        if not f.documentation.strip(): iss.append("empty HELP text")
        hc = {l for s in f.samples for l in s.labels} & HIGH_CARD
        if hc: iss.append(f"high-cardinality label(s): {sorted(hc)}")
        if iss: raise Warn("; ".join(iss))

## 4) الـ HTTP metrics: detection + دقة الـ counter
بنبعت **N** requests معروفة وبنقارن الـ delta مع N بالظبط.

In [ ]:
D = detect_http(FAMS); display(pd.Series(D, name="auto-detected"))
rec.check("HTTP request counter detected", bool(D["counter"]), str(D["counter"]), "http")
rec.check("HTTP duration histogram detected", bool(D["hist"]), str(D["hist"]), "http")
rec.check("in-progress gauge detected", bool(D["gauge"]), str(D["gauge"]), "http", level="warn")

def pick_probe():
    if api.routes["health"][1]: return api.routes["health"][1]
    for p, ops in app.openapi()["paths"].items():
        if "get" in ops and "{" not in p and not re.search(r"docs|redoc|openapi|metrics", p): return p
PROBE = pick_probe(); print("probe path:", PROBE)
PL, SL = D["path_label"], D["status_label"]

def snap(): return parse_metrics(scrape(api).text)[1]
def by_class(flat):
    out = defaultdict(float)
    for (n, lab), v in flat.items():
        if n == D["counter"]: out[str(dict(lab).get(SL, "?"))[0] + "xx"] += v
    return out

In [ ]:
N = 7
with rec.case(f"counter increases by EXACTLY {N} after {N} requests to {PROBE}", "http-counter") as c:
    need(D["counter"] and PROBE)
    b = snap()
    for _ in range(N): assert api.call("GET", PROBE).status_code == 200
    a = snap(); kw = {PL: PROBE} if PL else {}
    d = sel(a, D["counter"], **kw) - sel(b, D["counter"], **kw)
    assert d == N, f"expected +{N}, got +{d} for {PL}={PROBE!r} - is this path excluded from instrumentation, or labelled differently?"

with rec.case("status label reflects 2xx / 4xx classes", "http-counter") as c:
    need(D["counter"] and SL)
    b = by_class(snap())
    api.call("GET", PROBE)                                        # 2xx
    api.call("GET", "/__no_such_page__/" + _uuid.uuid4().hex)     # 404
    api.call("GET", "/__no_such_page__/" + _uuid.uuid4().hex)     # 404
    if api.routes["login"][1]: api.call("POST", api.routes["login"][1], json={})   # 422
    a = by_class(snap()); d2, d4 = a["2xx"] - b["2xx"], a["4xx"] - b["4xx"]
    c.note(f"+2xx={d2:.0f}, +4xx={d4:.0f}")
    assert d2 >= 1, "2xx not counted"; assert d4 >= 2, "4xx (404/422) not counted - unmatched routes may be skipped by the middleware"

with rec.case("exceptions / 500s are counted (5xx)", "http-counter") as c:
    need(D["counter"] and SL)
    from fastapi import FastAPI
    probe_app_path = "/__nb_boom__"
    if probe_app_path not in [r.path for r in app.routes]:
        @app.get(probe_app_path, include_in_schema=False)
        def _boom(): raise RuntimeError("intentional error from notebook 06")
    b = by_class(snap()); r = api.call("GET", probe_app_path); a = by_class(snap())
    c.note(f"HTTP {r.status_code}, +5xx={a['5xx'] - b['5xx']:.0f}")
    assert r.status_code == 500
    assert a["5xx"] - b["5xx"] >= 1, "an unhandled exception returned 500 but was NOT counted as 5xx"

## 5) الـ Histogram: صحة الرياضيات والوحدات

In [ ]:
def hist_groups(flat, name):
    g = defaultdict(lambda: {"b": []})
    for (n, lab), v in flat.items():
        d = dict(lab)
        if n == name + "_bucket":
            le = d.pop("le"); g[tuple(sorted(d.items()))]["b"].append((float("inf") if le == "+Inf" else float(le), v))
        elif n == name + "_count": g[tuple(sorted(d.items()))]["count"] = v
        elif n == name + "_sum": g[tuple(sorted(d.items()))]["sum"] = v
    return g

HG = hist_groups(snap(), D["hist"]) if D["hist"] else {}
with rec.case("buckets are cumulative (non-decreasing) in every series", "histogram") as c:
    need(HG or None)
    bad = [k for k, g in HG.items() if [v for _, v in sorted(g["b"])] != sorted(v for _, v in g["b"])]
    assert not bad, f"{len(bad)} series with decreasing buckets, e.g. {bad[:1]}"; c.note(f"{len(HG)} series checked")
with rec.case("the +Inf bucket equals _count", "histogram"):
    need(HG or None)
    bad = [k for k, g in HG.items() if g["b"] and "count" in g and max(g["b"])[1] != g["count"]]
    assert not bad, f"{len(bad)} series where +Inf != count"
with rec.case("_sum is >= 0 and _count >= 0", "histogram"):
    need(HG or None); assert all(g.get("sum", 0) >= 0 and g.get("count", 0) >= 0 for g in HG.values())
with rec.case("histogram count grows by N and unit is SECONDS (not ms)", "histogram") as c:
    need(D["hist"] and PROBE)
    kw = {PL: PROBE} if PL else {}
    b = snap(); client_ms = []
    for _ in range(N):
        t0 = time.perf_counter(); api.call("GET", PROBE); client_ms.append((time.perf_counter() - t0) * 1000)
    a = snap()
    dc = sel(a, D["hist"] + "_count", **kw) - sel(b, D["hist"] + "_count", **kw)
    ds = sel(a, D["hist"] + "_sum", **kw) - sel(b, D["hist"] + "_sum", **kw)
    assert dc == N, f"count delta {dc} != {N}"
    server_mean_s, client_mean_s = ds / dc, statistics.mean(client_ms) / 1000
    c.note(f"server mean={server_mean_s * 1000:.2f}ms, client mean={client_mean_s * 1000:.2f}ms")
    assert 0 < server_mean_s < 5, f"mean {server_mean_s} - looks like milliseconds instead of seconds"
    assert server_mean_s <= client_mean_s * 1.3 + 0.002, "server-side time is larger than what the client measured -> wrong timer scope/unit"

## 6) Label cardinality (أخطر مشكلة في الـ metrics)
لو الـ path label فيه UUID خام، كل request بيعمل time-series جديدة → الـ Prometheus بيتخنق والـ memory بتكبر.

In [ ]:
def series_count(): return len(series_keys(snap(), D["counter"])) if D["counter"] else 0

with rec.case("templated route with 6 different UUIDs creates NO new series", "cardinality") as c:
    need(D["counter"]); tpl = api.routes["file_get"][1]; need(tpl)
    api.call("GET", fill(tpl, _uuid.uuid4())); s0 = series_count()
    for _ in range(6): api.call("GET", fill(tpl, _uuid.uuid4()))
    s1 = series_count(); c.note(f"series {s0} -> {s1}")
    assert s1 - s0 <= CONFIG["THRESHOLDS"]["max_series_growth_per_new_path"], f"+{s1 - s0} series: the raw path is used as a label instead of the route template ('/files/{{id}}')"

with rec.case("6 random UNKNOWN paths (404) create NO new series", "cardinality") as c:
    need(D["counter"]); api.call("GET", "/__x__/" + _uuid.uuid4().hex); s0 = series_count()
    for _ in range(6): api.call("GET", "/__x__/" + _uuid.uuid4().hex)
    s1 = series_count(); c.note(f"series {s0} -> {s1}")
    if s1 - s0 > 0: raise Warn(f"+{s1 - s0} series: scanners hitting random URLs can blow up your Prometheus. Map unmatched paths to one label (e.g. 'none')")

with rec.case("no path label contains a raw UUID / numeric id", "cardinality") as c:
    need(PL)
    vals = {dict(lab).get(PL) for (n, lab) in snap() if n == D["counter"]} - {None}
    bad = [v for v in vals if re.search(r"[0-9a-f]{8}-[0-9a-f]{4}-|/\d{3,}(/|$)", v)]
    c.note(f"{len(vals)} distinct {PL} values"); assert not bad, f"raw ids in label: {bad[:3]}"

with rec.case("total series count is reasonable (< 2000)", "cardinality") as c:
    n = len(snap()); c.note(f"{n} series"); 
    if n > 2000: raise Warn(f"{n} series")

## 7) تحت الـ Concurrency
20 thread × 5 requests = 100 request متزامنة. الـ counter لازم يزيد 100 بالظبط، والـ in-progress gauge لازم يرجع للقيمة الأصلية.

In [ ]:
from fastapi.testclient import TestClient
def worker(_):
    cl = TestClient(app, raise_server_exceptions=False); codes = []
    for _ in range(5): codes.append(cl.get(PROBE).status_code)
    return codes

with rec.case("100 concurrent requests are counted exactly (thread-safety)", "concurrency") as c:
    need(D["counter"] and PROBE)
    kw = {PL: PROBE} if PL else {}; b = snap()
    t0 = time.perf_counter()
    with ThreadPoolExecutor(20) as ex: codes = [x for l in ex.map(worker, range(20)) for x in l]
    c.note(f"{len(codes)} requests in {time.perf_counter() - t0:.2f}s, statuses={sorted(set(codes))}")
    a = snap(); d = sel(a, D["counter"], **kw) - sel(b, D["counter"], **kw)
    assert d == len(codes), f"counter +{d} but {len(codes)} requests were served"
with rec.case("in-progress gauge returns to baseline after the load", "concurrency") as c:
    need(D["gauge"]); v = sel(snap(), D["gauge"]); c.note(f"value={v}"); assert v == 0, f"gauge stuck at {v} -> leaked in-flight counter (missing finally/dec)"

## 8) Business metrics بعد flow كامل
بنلقط snapshot، نشغّل `register → upload → ingest → search → chat`، ونقارن. أي metric مخصصة ما اتحركتش = غالباً مش متوصّلة بالكود.

In [ ]:
before = snap()
FLOW = None
with rec.case("run a full flow (upload -> ingest -> search -> chat) while measuring", "business") as c:
    FLOW = quick_flow(api, files=("orion_manual.txt", "blue_nile_plan_ar.txt")); c.note({k: round(v, 1) for k, v in FLOW["timings"].items()})
after = snap()

with rec.case("counters/histograms never go backwards (no unexpected resets)", "business") as c:
    cnt = {s.name for f in parse_metrics(scrape(api).text)[0] if f.type in ("counter", "histogram", "summary") for s in f.samples}
    back = [(k[0], before[k], after[k]) for k in before if k[0] in cnt and k in after and after[k] < before[k] and not k[0].endswith(("_sum", "_created"))]
    assert not back, f"decreased: {back[:3]}"

CH = diff_flat(before, after)
is_default = CH["sample"].str.startswith(DEFAULT_PREFIXES) if len(CH) else pd.Series(dtype=bool)
is_http = CH["sample"].str.startswith("http") if len(CH) else pd.Series(dtype=bool)
print("Custom (business) series that changed during the flow:")
display(CH[~is_default & ~is_http].reset_index(drop=True) if len(CH) else CH)
print("HTTP series that changed (top 12):")
display(CH[is_http].sort_values("delta", ascending=False).head(12).reset_index(drop=True) if len(CH) else CH)

In [ ]:
# ---- ADAPT HERE: put the business metrics YOU expect, as  sample_name: minimum_delta
BUSINESS_EXPECT = {
    # "rag_files_ingested_total": 2,
    # "rag_chunks_created_total": 2,
    # "rag_embedding_duration_seconds_count": 1,
    # "rag_search_requests_total": 1,
    # "rag_llm_requests_total": 1,
}
for name, mn in BUSINESS_EXPECT.items():
    d = sel(after, name) - sel(before, name)
    rec.check(f"{name} increased by >= {mn}", d >= mn, f"delta={d}", "business")
if not BUSINESS_EXPECT: print("BUSINESS_EXPECT is empty -> using automatic detection below.")

moved = set(CH["sample"].str.replace(r"(_total|_count|_sum|_bucket)$", "", regex=True)) if len(CH) else set()
for f in FAMS:
    if f.name.startswith(DEFAULT_PREFIXES) or f.name.startswith("http"): continue
    with rec.case(f"custom metric is actually used: {f.name}", "business") as c:
        if f.name not in moved:
            raise Warn(f"never changed during a full ingest+search+chat flow ({len(f.samples)} series) -> not instrumented, or only used on another code path")
        c.note("changed")

## 9) Ops checks

In [ ]:
with rec.case(f"/metrics scrape is fast (p95 < {CONFIG['THRESHOLDS']['scrape_p95_ms']} ms)", "ops") as c:
    ts = []
    for _ in range(30):
        t0 = time.perf_counter(); rr = scrape(api); ts.append((time.perf_counter() - t0) * 1000)
    c.note(f"p50={pct(ts, 50):.0f}ms p95={pct(ts, 95):.0f}ms, size={len(rr.content) / 1024:.1f}KB")
    assert pct(ts, 95) < CONFIG["THRESHOLDS"]["scrape_p95_ms"]
with rec.case("/metrics is reachable without authentication (what Prometheus needs)", "ops") as c:
    r = api.call("GET", CONFIG["METRICS_PATH"]); assert r.status_code == 200
    raise Warn("it is public: in production expose it only on an internal network / port, or behind basic-auth at the proxy")
with rec.case("multi-process safety (PROMETHEUS_MULTIPROC_DIR)", "ops") as c:
    workers = os.environ.get("WEB_CONCURRENCY") or os.environ.get("UVICORN_WORKERS")
    mp = os.environ.get("PROMETHEUS_MULTIPROC_DIR")
    if workers and int(workers) > 1 and not mp:
        raise AssertionError(f"{workers} workers but PROMETHEUS_MULTIPROC_DIR is not set -> every scrape returns a random worker's partial numbers")
    if not mp: raise Warn("single-process OK; if you ever run `uvicorn --workers N` / gunicorn you must set PROMETHEUS_MULTIPROC_DIR")

## 10) سلامة الـ `prometheus_client` نفسها (sanity)
اختبارات صغيرة على registry معزول عشان نتأكد من فهمنا لسلوك الـ Counter/Gauge/Histogram.

In [ ]:
from prometheus_client import CollectorRegistry, Counter, Gauge, Histogram, generate_latest

reg = CollectorRegistry()
cn = Counter("t_requests", "test counter", ["route"], registry=reg)
gg = Gauge("t_inflight", "test gauge", registry=reg)
hh = Histogram("t_latency_seconds", "test hist", buckets=(0.1, 0.5, 1.0), registry=reg)

with rec.case("Counter only goes up (inc(-1) raises)", "prom-client"):
    cn.labels("a").inc(); cn.labels("a").inc(2)
    try: cn.labels("a").inc(-1); raise AssertionError("negative inc accepted")
    except ValueError: pass
    assert reg.get_sample_value("t_requests_total", {"route": "a"}) == 3
with rec.case("Gauge inc/dec/set", "prom-client"):
    gg.inc(); gg.inc(); gg.dec(); assert reg.get_sample_value("t_inflight") == 1; gg.set(7); assert reg.get_sample_value("t_inflight") == 7
with rec.case("Histogram puts values in the right cumulative buckets", "prom-client"):
    for v in (0.05, 0.3, 0.3, 2.0): hh.observe(v)
    g = lambda le: reg.get_sample_value("t_latency_seconds_bucket", {"le": le})
    assert (g("0.1"), g("0.5"), g("1.0"), g("+Inf")) == (1, 3, 3, 4), (g("0.1"), g("0.5"), g("1.0"), g("+Inf"))
    assert abs(reg.get_sample_value("t_latency_seconds_sum") - 2.65) < 1e-9
with rec.case("exposition output round-trips through the parser", "prom-client"):
    fams, flat = parse_metrics(generate_latest(reg).decode()); assert {f.name for f in fams} == {"t_requests", "t_inflight", "t_latency_seconds"}
with rec.case("registering the same metric name twice fails (guards against double-import bugs)", "prom-client"):
    try: Counter("t_requests", "dup", registry=reg); raise AssertionError("duplicate registration was accepted")
    except ValueError: pass
with rec.case("label values with quotes / newlines / unicode are escaped correctly", "prom-client"):
    cn.labels('he said "hi"\nمرحبا').inc(); fams, flat = parse_metrics(generate_latest(reg).decode())
    assert any('مرحبا' in dict(k[1]).get("route", "") for k in flat), "unicode label lost"

## 11) رسومات

In [ ]:
fin = snap(); fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
if D["counter"] and SL:
    cls = defaultdict(float)
    for (n, lab), v in fin.items():
        if n == D["counter"]: cls[str(dict(lab).get(SL))[0] + "xx"] += v
    ax[0].bar(list(cls), list(cls.values()), color=["#59a14f" if k.startswith("2") else "#edc948" if k.startswith("4") else "#e15759" for k in cls]); ax[0].set_title("Requests by status class")
if D["counter"] and PL:
    top = defaultdict(float)
    for (n, lab), v in fin.items():
        if n == D["counter"]: top[dict(lab).get(PL)] += v
    top = dict(sorted(top.items(), key=lambda x: -x[1])[:8]); ax[1].barh(list(top)[::-1], list(top.values())[::-1]); ax[1].set_title("Top paths by request count")
if D["hist"]:
    agg = defaultdict(float)
    for (n, lab), v in fin.items():
        if n == D["hist"] + "_bucket": agg[dict(lab)["le"]] += v
    les = sorted(agg, key=lambda x: float("inf") if x == "+Inf" else float(x)); cum = [agg[l] for l in les]
    per = [cum[0]] + [cum[i] - cum[i - 1] for i in range(1, len(cum))]
    ax[2].bar(les, per); ax[2].set_title("Latency distribution (per bucket, seconds)"); ax[2].tick_params(axis="x", rotation=60)
plt.tight_layout(); plt.show()

In [ ]:
rec.summary(); rec.table(); rec.save()